# Horizon predictions for the unconstrained prompts

Runs the saved pipeline over `data/no_constraints_activation_pls_projection.csv`
and reports a table of predictions and the points projected onto the fitted
surface. Nothing is fitted except the folder's own registration shift, which
uses no horizon labels.

These 37 rows are one per task, with the time quantity removed from the prompt,
so **there is no ground truth here** -- `time_horizon_months` is empty for every
row. Nothing below is scored; the prediction is the model's implicit read of how
long each task takes when the prompt does not say.

Two caveats specific to this file, both of which the notebook makes visible
rather than hiding:

* `no_constraints` is **not** one of the folders the shift table was fitted on.
  It is registered here from its own coordinates, which needs no labels, but the
  resulting shift is fitted on only 37 points and is correspondingly less
  settled than the four in the table.
* Every training point is an average over dozens of prompt variants sharing a
  task and a horizon; these rows are far less averaged, so they sit further from
  the surface and are predicted less accurately.

## Setup

In [ ]:
# Reload edited modules automatically: the helpers under `vendor/` change often,
# and without this an already-imported module keeps its stale copy for the life
# of the kernel (an ImportError for a function that plainly exists on disk).
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.graph_objects as go

# The saved models unpickle classes from `temporal_manifolds`; a minimal copy of
# that code lives in `vendor/` at the repo root.
REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "vendor").is_dir():
    REPO_ROOT = REPO_ROOT.parent
if str(REPO_ROOT / "vendor") not in sys.path:
    sys.path.insert(0, str(REPO_ROOT / "vendor"))

from temporal_manifolds.viz.extruded_surface import load_surface_model
from utils.horizon_classes import (
    HORIZON_CLASS_LABELS,
    MERGED_HORIZON_CLASS_LABELS,
    merge_horizon_classes,
)
from utils.ordinal_regression import load_ordinal_model

print("repo root:", REPO_ROOT)

## Configuration

`no_constraints` carries no `abst` rows, so only the main classifier is needed --
the one that keeps the reconstruction-residual PCs.

In [ ]:
import joblib

INPUT_CSV = REPO_ROOT / "data" / "no_constraints_activation_pls_projection.csv"

OFFSET_DEGREE = 2   # extrusion profile degree, picks the surface artifact
DEGREE = 2          # classifier polynomial degree, picks the model artifact

SURFACE_PATH = (
    REPO_ROOT / "models"
    / f"ctype_only_activation_surface_PLS1-PLS2-by-t_extruded-PLS3_degree-{OFFSET_DEGREE}.joblib"
)
SHIFT_TABLE_PATH = REPO_ROOT / "models" / "new_per_folder_shift_table.joblib"
CLASSIFIER_PATH = (
    REPO_ROOT / "models"
    / f"new_horizon_class_main_ordinal_binary-decomposition_degree-{DEGREE}.joblib"
)

COORDINATE_COLUMNS = ["PLS1", "PLS2", "PLS3"]
PARAMETER_SAMPLES = 2000   # density of the nearest-t search when projecting

# How to register this file against the surface:
#   "fit"    -- fit a shift from these rows' own coordinates (no labels used)
#   "nearest"-- reuse whichever existing folder's shift fits best
#   "none"   -- no shift at all
SHIFT_MODE = "fit"

surface, surface_metadata = load_surface_model(SURFACE_PATH)
shift_table = joblib.load(SHIFT_TABLE_PATH)
SHIFTS = {name: np.asarray(value, dtype=float) for name, value in shift_table["shifts"].items()}
classifier, classifier_features, classifier_metadata = load_ordinal_model(CLASSIFIER_PATH)

print("surface   :", SURFACE_PATH.name,
      "| t range", np.round(surface.training_parameter_bounds, 3))
print("shifts    :", SHIFT_TABLE_PATH.name, "->", sorted(SHIFTS))
print("classifier:", CLASSIFIER_PATH.name, "->", classifier_features)
print("shift mode:", SHIFT_MODE)

## Read the input

In [ ]:
raw = pd.read_csv(INPUT_CSV)

required = COORDINATE_COLUMNS + [c for c in classifier_features if c not in {"t", "u"}]
missing = [column for column in required if column not in raw.columns]
if missing:
    raise KeyError(f"{INPUT_CSV.name} is missing required column(s): {missing}")

usable = raw[required].notna().all(axis=1)
df = raw[usable].copy().reset_index(drop=True)

labelled = int(raw.get("time_horizon_months", pd.Series(dtype=float)).notna().sum())
print(f"{len(raw)} row(s) read, {len(df)} usable, {int((~usable).sum())} dropped")
print(f"rows carrying a ground-truth horizon: {labelled}"
      + ("  -- nothing below is scored" if labelled == 0 else ""))
df[["task", "source_folder"] + COORDINATE_COLUMNS].head()

## Register the folder

The shift is the rigid displacement of this folder's whole point cloud in the
PLS1-PLS2 plane, the plane orthogonal to `u`. It is fitted by alternating least
squares -- assign each point its nearest `t`, move the curve to the mean in-plane
residual, repeat -- and the objective is geometric distance only, so **no horizon
labels are involved**. That is what makes registering an unlabelled folder
legitimate rather than circular.

All three modes are evaluated so the choice is visible; `SHIFT_MODE` selects
which one the rest of the notebook uses.

In [ ]:
def fit_shift(points, iterations=40, tolerance=1e-9):
    """Fit one in-plane translation by alternating least squares."""

    shift = np.zeros(2)
    for _ in range(iterations):
        moved = points.copy()
        moved[:, :2] -= shift
        t, u, _ = surface.project(moved, parameter_samples=PARAMETER_SAMPLES)
        step = (moved[:, :2] - surface.predict(t, u)[:, :2]).mean(axis=0)
        if np.abs(step).max() < tolerance:
            break
        shift = shift + step
    return shift


def rmse_under(points, shift):
    moved = points.copy()
    moved[:, :2] -= shift
    return float(np.sqrt((surface.project(moved, parameter_samples=PARAMETER_SAMPLES)[2] ** 2).mean()))


points = df[COORDINATE_COLUMNS].to_numpy(float)
own_shift = fit_shift(points)

options = {"none": np.zeros(2), "fit": own_shift}
for name, value in SHIFTS.items():
    options[f"reuse {name}"] = value

comparison = pd.DataFrame([
    {"option": name, "dPLS1": value[0], "dPLS2": value[1],
     "surface_rmse": rmse_under(points, value)}
    for name, value in options.items()
]).sort_values("surface_rmse")
print(comparison.round(3).to_string(index=False))

if SHIFT_MODE == "fit":
    shift = own_shift
elif SHIFT_MODE == "nearest":
    best = min(SHIFTS, key=lambda name: rmse_under(points, SHIFTS[name]))
    shift = SHIFTS[best]
    print(f"\nnearest existing folder: {best}")
elif SHIFT_MODE == "none":
    shift = np.zeros(2)
else:
    raise ValueError(f"unknown SHIFT_MODE: {SHIFT_MODE!r}")

print(f"\nusing shift ({shift[0]:.3f}, {shift[1]:.3f}) -> "
      f"surface RMSE {rmse_under(points, shift):.3f}")

## Project and predict

`u` is exactly PLS3; `t` is the nearest point along the extruded curve once the
shift has been removed.

In [ ]:
shifted = points.copy()
shifted[:, :2] -= shift

t, u, distance = surface.project(shifted, parameter_samples=PARAMETER_SAMPLES)
df["t"], df["u"], df["surface_distance"] = t, u, distance

predicted = classifier.predict(df[classifier_features].to_numpy(float))
df["horizon_class_predicted"] = predicted
df["horizon_class_label_predicted"] = np.array(HORIZON_CLASS_LABELS, dtype=object)[predicted]

merged = merge_horizon_classes(predicted)
df["horizon_class_label_merged_predicted"] = np.array(
    MERGED_HORIZON_CLASS_LABELS, dtype=object)[merged]

# Class probabilities, and the cumulative log-odds behind them. This family has
# no per-class logits: column k is the "is the class greater than k?" model.
probabilities = classifier.predict_proba(df[classifier_features].to_numpy(float))
total = probabilities.sum(axis=1, keepdims=True)
normalized = np.divide(probabilities, np.where(total > 0, total, 1.0))
df["predicted_probability"] = normalized.max(axis=1)

inner = classifier[:-1].transform(df[classifier_features].to_numpy(float))
logits = classifier.named_steps["ordinal"].decision_function(inner)
df["logits_monotone"] = ~(np.diff(logits, axis=1) > 0).any(axis=1)

print("distance to surface: mean %.3f, max %.3f" % (distance.mean(), distance.max()))
outside = (t < surface.training_parameter_bounds[0]) | (t > surface.training_parameter_bounds[1])
print("points projecting outside the fitted t range:", int(outside.sum()))
print("rows with non-monotone cumulative logits: %d of %d" % (int((~df.logits_monotone).sum()), len(df)))
print("mean predicted-class probability: %.4f" % df["predicted_probability"].mean())

## Table of predictions

Ordered by `t`, so the table reads from the shortest implied horizon to the
longest. `surface_distance` is what the projection threw away -- a large value
means `(t, u)` is a poor summary of where that prompt actually sits, so treat
its row with more suspicion.

In [ ]:
table = (
    df[["task", "t", "u", "surface_distance",
        "horizon_class_predicted", "horizon_class_label_predicted",
        "horizon_class_label_merged_predicted",
        "predicted_probability", "logits_monotone"]]
    .sort_values("t")
    .reset_index(drop=True)
)
table.index.name = "rank"

with pd.option_context("display.max_rows", None, "display.width", 200,
                       "display.max_colwidth", 46):
    print(table.round(3).to_string())

print("\npredicted class counts (9-class):")
print(df["horizon_class_label_predicted"].value_counts()
        .reindex(HORIZON_CLASS_LABELS, fill_value=0).to_string())
print("\npredicted class counts (7-class):")
print(df["horizon_class_label_merged_predicted"].value_counts()
        .reindex(MERGED_HORIZON_CLASS_LABELS, fill_value=0).to_string())

table

## The surface, and the points projected onto it

The sheet is the fitted extruded surface. The markers are **not** the rows' PLS
coordinates: each is `S(t, u)`, the nearest point *on* the surface, so every
marker lies exactly on the sheet by construction. Hover carries the task, the
discarded distance, and the predicted class. Everything is drawn in the shifted
frame, which is the frame the surface lives in.

In [ ]:
projected = surface.predict(df["t"].to_numpy(float), df["u"].to_numpy(float))
df[["PLS1_on_surface", "PLS2_on_surface", "PLS3_on_surface"]] = projected

u_values = df["u"].to_numpy(float)
u_lo, u_hi = float(np.min(u_values)), float(np.max(u_values))
if not np.isfinite([u_lo, u_hi]).all() or (u_hi - u_lo) < 1e-6:
    u_lo, u_hi = (float(v) for v in surface.training_extrusion_bounds)
pad = 0.15 * max(u_hi - u_lo, 1.0)
u_grid = np.linspace(u_lo - pad, u_hi + pad, 60)
t_grid = np.linspace(*surface.training_parameter_bounds, 200)

grid_x, grid_y, grid_z = surface.grid(t_grid, u_grid)

figure = go.Figure()
figure.add_surface(
    x=grid_x, y=grid_y, z=grid_z,
    surfacecolor=np.broadcast_to(t_grid[:, None], grid_x.shape),
    colorscale="Viridis", opacity=0.55, showscale=True,
    colorbar={"title": "t"},
    name="fitted surface",
    hovertemplate="t=%{surfacecolor:.3f}<br>PLS1=%{x:.2f}<br>PLS2=%{y:.2f}<br>PLS3=%{z:.2f}<extra></extra>",
)
figure.add_scatter3d(
    x=df["PLS1_on_surface"], y=df["PLS2_on_surface"], z=df["PLS3_on_surface"],
    mode="markers",
    marker={"size": 6, "color": df["t"], "colorscale": "Plasma",
            "line": {"width": 1, "color": "white"}},
    name="no_constraints prompts, projected",
    customdata=np.column_stack([
        df["task"], df["t"], df["u"], df["surface_distance"],
        df["horizon_class_label_predicted"], df["predicted_probability"],
    ]),
    hovertemplate=(
        "%{customdata[0]}<br>t=%{customdata[1]:.3f}  u=%{customdata[2]:.2f}"
        "<br>distance to surface=%{customdata[3]:.2f}"
        "<br>predicted: %{customdata[4]} (p=%{customdata[5]:.2f})<extra></extra>"
    ),
)
figure.update_layout(
    scene={"xaxis_title": "PLS1 (shifted)",
           "yaxis_title": "PLS2 (shifted)",
           "zaxis_title": "PLS3 = u"},
    title="no_constraints prompts projected onto the fitted surface",
    autosize=True, width=None, height=700,
    legend={"orientation": "h", "yanchor": "bottom", "y": -0.08},
)
figure.show()

print("distance discarded by the projection: mean %.3f, max %.3f"
      % (df["surface_distance"].mean(), df["surface_distance"].max()))